# Song Popularity Analysis

Question: **What makes songs popular?**

This notebook analyzes the Kaggle dataset `900k Spotify`. It defines popularity before comparing drivers, uses the top 10% of tracks by the selected target as the primary popular segment, and separates audio-feature drivers from artist/album metadata drivers.

## Data Source And Import

Dataset: https://www.kaggle.com/datasets/devdope/900k-spotify

Place or symlink `spotify_dataset.csv` in `data/raw/`. Raw files larger than 10 MB should not be committed. No YouTube or video deliverable is required for this project.

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path('src').resolve()))

import pandas as pd

from song_popularity.data import (
    DataNotFoundError,
    DEFAULT_ANALYSIS_SOURCE_COLUMNS,
    TAG_SOURCE_COLUMNS,
    column_inventory,
    discover_raw_data_files,
    duplicate_summary,
    full_valid_dataset,
    load_table,
    normalize_analysis_columns,
    validate_audio_feature_ranges,
    write_processed_table,
)
from song_popularity.popularity import (
    ACTIVITY_TAG_COLUMNS,
    AUDIO_FEATURE_COLUMNS,
    activity_tag_lift,
    audio_feature_comparison,
    available_target_candidates,
    categorical_tag_lift,
    metadata_exposure_comparison,
    popular_tag_venn_summary,
    rank_drivers,
    select_popularity_target,
    top_decile_segment,
)
from song_popularity.plots import ranked_drivers as ranked_drivers_chart, save_figure, segment_comparison, tag_venn_chart, target_distribution

pd.set_option('display.max_columns', 100)

In [ ]:
try:
    raw_files = discover_raw_data_files('data/raw')
    data_path = raw_files[0]
    print(f'Loading {data_path}')
    source_columns = list(dict.fromkeys(list(DEFAULT_ANALYSIS_SOURCE_COLUMNS) + list(TAG_SOURCE_COLUMNS)))
    df = normalize_analysis_columns(load_table(data_path, columns=source_columns))
except DataNotFoundError as exc:
    print(exc)
    df = pd.DataFrame()

df.shape

## Field Inventory And Data Quality

In [ ]:
if df.empty:
    print('Dataset not loaded yet. Download it from Kaggle and place it in data/raw/.')
else:
    inventory = column_inventory(df)
    duplicates = duplicate_summary(df)
    audio_ranges = validate_audio_feature_ranges(df)
    display(inventory.head(50))
    display(duplicates)
    display(audio_ranges)

## Popularity Target Definition

Preference order:
1. Direct popularity field, including this dataset's `Popularity` column.
2. Playlist-derived proxy.
3. Artist repetition/counts as a **weak proxy** if no better target exists.

In [ ]:
if df.empty:
    target = None
    segmented = pd.DataFrame()
    segment_counts = pd.DataFrame()
    print('Target selection skipped until data is available.')
else:
    display(available_target_candidates(df))
    df_with_target, target = select_popularity_target(df)
    segmented, segment_counts = top_decile_segment(df_with_target, target)
    print(target)
    display(segment_counts)
    if target.target_strength == 'weak_proxy':
        print('Caveat: artist repetition/counts are a weak proxy for popularity.')

## Full Valid Dataset

Final reported findings must use the full valid dataset after documented cleaning/exclusion rules. Sampling may be used only for exploratory development or performance checks.

In [ ]:
if df.empty or target is None:
    valid = pd.DataFrame()
    exclusion_summary = pd.DataFrame()
    print('Full-valid-dataset filtering skipped until data and target are available.')
else:
    required = [target.target_value_column] + [c for c in AUDIO_FEATURE_COLUMNS if c in segmented.columns]
    valid, exclusion_summary = full_valid_dataset(segmented, required)
    display(exclusion_summary)
    print(f'Final findings will use {len(valid):,} valid records.')

## Popularity Drivers

In [ ]:
if valid.empty:
    audio_comparison = pd.DataFrame()
    metadata_comparison = pd.DataFrame()
    ranked = pd.DataFrame()
    print('Driver analysis skipped until valid data is available.')
else:
    audio_comparison = audio_feature_comparison(valid)
    metadata_comparison = metadata_exposure_comparison(valid)
    ranked = rank_drivers(audio_comparison, metadata_comparison)
    display(audio_comparison.head(10))
    display(metadata_comparison.head(10))
    display(ranked)

## Popularity Tags

In [ ]:
if valid.empty:
    activity_tags = pd.DataFrame()
    genre_tags = pd.DataFrame()
    emotion_tags = pd.DataFrame()
    explicit_tags = pd.DataFrame()
    top_venn_tags = pd.DataFrame()
    tag_venn_regions = pd.DataFrame()
    print('Tag analysis skipped until valid data is available.')
else:
    activity_tags = activity_tag_lift(valid, ACTIVITY_TAG_COLUMNS)
    genre_tags = categorical_tag_lift(valid, 'genre')
    emotion_tags = categorical_tag_lift(valid, 'emotion')
    explicit_tags = categorical_tag_lift(valid, 'explicit')
    top_venn_tags, tag_venn_regions = popular_tag_venn_summary(valid)
    display(activity_tags)
    display(genre_tags.head(15))
    display(emotion_tags)
    display(explicit_tags)
    display(top_venn_tags)
    display(tag_venn_regions)

## Figures

In [ ]:
if not valid.empty and target is not None:
    save_figure(target_distribution(valid, target.target_value_column), 'figures/target_distribution.html')
    if not audio_comparison.empty:
        save_figure(segment_comparison(audio_comparison.head(10), 'Audio Feature Differences'), 'figures/audio_feature_differences.html')
    if not ranked.empty:
        save_figure(ranked_drivers_chart(ranked), 'figures/ranked_drivers.html')
    if not top_venn_tags.empty and not tag_venn_regions.empty:
        save_figure(tag_venn_chart(top_venn_tags, tag_venn_regions, 'Top 3 Tags On Popular Songs'), 'figures/popular_tag_venn.html')
    print('Figures exported to figures/.')
else:
    print('Figure export skipped until valid data is available.')

## Findings And Limitations

Replace this section with evidence-backed findings after running the notebook with the Kaggle data.

Required caveats:
- Popularity may be direct evidence, a playlist-derived proxy, or a weak artist-repetition proxy depending on available fields.
- Playlist-derived popularity reflects playlist behavior and dataset construction, not universal listener preference.
- Artist exposure, genre mix, release timing, and platform behavior may affect observed popularity.
- The analysis is associative, not causal.